# Ekstraksi Data CO dan SO₂ — Kecamatan Kedungpring

**Mata Kuliah:** Proyek Sains Data — Semester 5
**Wilayah:** Kecamatan Kedungpring, Kabupaten Lamongan
**Polutan:** CO (Karbon Monoksida) dan SO₂ (Sulfur Dioksida)
**Rentang Waktu:** 31 Agustus 2025 — 31 Agustus 2026

## Tujuan Notebook

Notebook ini menarik data Sentinel-5P L2 untuk dua polutan sekaligus — **CO** dan
**SO₂** — menggunakan struktur proses yang identik dengan yang sudah dipakai untuk
NO₂ sebelumnya. Karena kedua polutan diproses dengan langkah yang sama persis, seluruh
proses ditulis sekali dalam bentuk fungsi, lalu dijalankan berulang (*loop*) untuk
`["CO", "SO2"]`.

Alur pada notebook ini:
1. Terhubung ke server openEO Copernicus Data Space (device code flow).
2. Menentukan AOI Kecamatan Kedungpring (sama seperti proses NO₂).
3. Untuk setiap polutan dalam `["CO", "SO2"]`: `load_collection` → agregasi temporal
   (harian) & spasial (AOI) → download NetCDF → konversi ke CSV.

➡️ **Tahap selanjutnya** (preprocessing, deteksi outlier, imputasi, dan ekstraksi 68
fitur TSFEL) ada di notebook terpisah:
`preprocessing-fitur-co-so2-kedungpring.ipynb`.

## Import Pustaka

In [1]:
import os
import openeo
import xarray as xr
import pandas as pd

NC_DIR = "../data/nc/"
CSV_DIR = "../data/csv/"
os.makedirs(NC_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Folder output siap:")
print(" -", os.path.abspath(NC_DIR))
print(" -", os.path.abspath(CSV_DIR))

Folder output siap:
 - d:\Semester 5\Proyek sain data\PSD\data\nc
 - d:\Semester 5\Proyek sain data\PSD\data\csv


## Koneksi & Otentikasi ke Copernicus Data Space

Menggunakan **OIDC Device Code Flow** — jalankan sel ini, buka tautan yang muncul,
login dengan akun Copernicus Data Space Ecosystem (CDSE).

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()
print("Terhubung sebagai:", connection.describe_account())

Authenticated using refresh token.
Terhubung sebagai: {'info': {'oidc_userinfo': {'email': 'triswanti1395@gmail.com', 'email_verified': True, 'family_name': "Jannatul Ma'wa", 'given_name': 'Triswanti', 'name': "Triswanti Jannatul Ma'wa", 'preferred_username': 'triswanti1395@gmail.com', 'sub': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}}, 'name': "Triswanti Jannatul Ma'wa", 'user_id': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}


## Area of Interest (AOI) & Daftar Polutan

AOI tetap sama seperti proses NO₂ sebelumnya — Kecamatan Kedungpring, Kabupaten
Lamongan. `POLLUTANTS_TO_PROCESS` mendaftar kedua polutan yang akan diproses dalam
loop: **CO** dan **SO2**.

In [3]:
KEDUNGPRING_BBOX = {
    "west": 112.1669,
    "south": -7.2075,
    "east": 112.2244,
    "north": -7.1386,
}

KEDUNGPRING_POLYGON = {
    "type": "Polygon",
    "coordinates": [[
        [112.1669, -7.1386],
        [112.2244, -7.1386],
        [112.2244, -7.2075],
        [112.1669, -7.2075],
        [112.1669, -7.1386],
    ]]
}

TEMPORAL_EXTENT = ["2025-08-31", "2026-08-31"]

POLLUTANTS_TO_PROCESS = ["CO", "SO2"]

print("AOI Kedungpring:", KEDUNGPRING_BBOX)
print("Polutan yang diproses:", POLLUTANTS_TO_PROCESS)

AOI Kedungpring: {'west': 112.1669, 'south': -7.2075, 'east': 112.2244, 'north': -7.1386}
Polutan yang diproses: ['CO', 'SO2']


## Fungsi Ekstraksi Satu Polutan

`extract_pollutant()` merangkum seluruh proses openEO: `load_collection` →
`aggregate_temporal_period` (rata-rata harian) → `aggregate_spatial` (rata-rata dalam
AOI Kedungpring) → `download` sebagai NetCDF. `nc_to_csv()` mengonversi hasil NetCDF
menjadi CSV, dengan deteksi kolom tanggal & nilai yang fleksibel (tidak bergantung
nama kolom pasti dari openEO — sama seperti versi final yang sudah diperbaiki pada
proses NO₂).

In [4]:
def extract_pollutant(connection, band, bbox, polygon, temporal_extent, output_dir):
    """Menarik satu polutan: agregasi harian (mean) + spasial (mean dalam polygon)."""
    datacube = connection.load_collection(
        "SENTINEL_5P_L2",
        spatial_extent=bbox,
        temporal_extent=temporal_extent,
        bands=[band],
    )
    daily_cube = datacube.aggregate_temporal_period(period="day", reducer="mean")
    spatial_result = daily_cube.aggregate_spatial(geometries=polygon, reducer="mean")

    nc_path = os.path.join(output_dir, f"{band.lower()}_kedungpring.nc")
    print(f"[{band}] Mengunduh ke: {nc_path} ... (mohon tunggu)")
    spatial_result.download(nc_path, format="netCDF")
    print(f"[{band}] Unduhan selesai.")
    return nc_path


def nc_to_csv(nc_path, band, csv_dir):
    """Konversi NetCDF -> CSV, dengan deteksi kolom tanggal & nilai yang fleksibel."""
    ds = xr.open_dataset(nc_path)
    df = ds.to_dataframe().reset_index()

    print(f"[{band}] Kolom hasil NetCDF:", df.columns.tolist())

    date_col = None
    for c in df.columns:
        try:
            converted = pd.to_datetime(df[c], errors="coerce")
            if converted.notna().sum() > 0:
                date_col = c
                break
        except Exception:
            pass

    if date_col is None:
        raise ValueError(f"[{band}] Kolom tanggal tidak ditemukan. Kolom: {df.columns.tolist()}")

    value_col = None
    for c in df.columns:
        if band.lower() in c.lower():
            value_col = c
            break
    if value_col is None:
        numeric_cols = [c for c in df.columns if c != date_col and pd.api.types.is_numeric_dtype(df[c])]
        if numeric_cols:
            value_col = numeric_cols[-1]

    if value_col is None:
        raise ValueError(f"[{band}] Kolom nilai tidak ditemukan. Kolom: {df.columns.tolist()}")

    print(f"[{band}] Kolom tanggal: {date_col} | Kolom nilai: {value_col}")

    df = df[[date_col, value_col]].rename(columns={date_col: "date", value_col: band})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df["date"] = df["date"].dt.date

    csv_path = os.path.join(csv_dir, f"{band}-Kedungpring.csv")
    df.to_csv(csv_path, index=False)
    print(f"[{band}] CSV disimpan: {csv_path} ({len(df)} baris)\n")
    return csv_path

## Menjalankan Ekstraksi untuk CO dan SO₂

Loop berikut memanggil `extract_pollutant()` lalu `nc_to_csv()` untuk setiap band
dalam `POLLUTANTS_TO_PROCESS`, dibungkus `try/except` agar kalau salah satu polutan
gagal diproses, polutan lainnya tetap bisa lanjut.

In [5]:
raw_csv_paths = {}

for band in POLLUTANTS_TO_PROCESS:
    try:
        nc_path = extract_pollutant(
            connection=connection,
            band=band,
            bbox=KEDUNGPRING_BBOX,
            polygon=KEDUNGPRING_POLYGON,
            temporal_extent=TEMPORAL_EXTENT,
            output_dir=NC_DIR,
        )
        raw_csv_paths[band] = nc_to_csv(nc_path, band, CSV_DIR)
    except Exception as e:
        print(f"[{band}] Gagal diproses: {e}\n")

print("Ringkasan file CSV mentah yang berhasil dibuat:")
for band, path in raw_csv_paths.items():
    print(f" - {band}: {path}")

[CO] Mengunduh ke: ../data/nc/co_kedungpring.nc ... (mohon tunggu)
[CO] Unduhan selesai.
[CO] Kolom hasil NetCDF: ['t', 'feature', 'CO', 'lat', 'lon', 'feature_names']
[CO] Kolom tanggal: t | Kolom nilai: CO
[CO] CSV disimpan: ../data/csv/CO-Kedungpring.csv (205 baris)

[SO2] Mengunduh ke: ../data/nc/so2_kedungpring.nc ... (mohon tunggu)
[SO2] Unduhan selesai.
[SO2] Kolom hasil NetCDF: ['t', 'feature', 'SO2', 'lat', 'lon', 'feature_names']
[SO2] Kolom tanggal: t | Kolom nilai: SO2
[SO2] CSV disimpan: ../data/csv/SO2-Kedungpring.csv (221 baris)

Ringkasan file CSV mentah yang berhasil dibuat:
 - CO: ../data/csv/CO-Kedungpring.csv
 - SO2: ../data/csv/SO2-Kedungpring.csv


## Verifikasi Hasil Ekstraksi

In [6]:
for band in POLLUTANTS_TO_PROCESS:
    if band not in raw_csv_paths:
        print(f"[{band}] Tidak tersedia (ekstraksi gagal).")
        continue
    df = pd.read_csv(raw_csv_paths[band])
    print(f"[{band}] Total baris: {len(df)} | Rentang tanggal: {df['date'].min()} s.d. {df['date'].max()}")

[CO] Total baris: 205 | Rentang tanggal: 2025-09-01 s.d. 2026-08-30
[SO2] Total baris: 221 | Rentang tanggal: 2025-08-31 s.d. 2026-08-30


## Ringkasan

| Item | Nilai |
| ---- | ----- |
| Wilayah | Kecamatan Kedungpring, Kabupaten Lamongan |
| Polutan | CO, SO₂ |
| Rentang waktu | 31 Agustus 2025 – 31 Agustus 2026 |
| Output | `../data/csv/CO-Kedungpring.csv`, `../data/csv/SO2-Kedungpring.csv` |

Data mentah pada tahap ini masih mengandung missing value (hari tanpa observasi
satelit) — akan ditangani pada notebook berikutnya.

➡️ **Lanjut ke:** `preprocessing-fitur-co-so2-kedungpring.ipynb`